# ControlNet vs ControlNet + IP-Adapter Experiment

This notebook compares Stable Diffusion + ControlNet depth guidance against Stable Diffusion + ControlNet + IP-Adapter.

Aim: test whether IP-Adapter improves image-based conditioning and visual consistency.

In [ ]:
!pip -q install -r /content/image-data-generation/requirements.txt

In [ ]:
import os
import time
import gc
import torch
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt

from diffusers import ControlNetModel, StableDiffusionControlNetImg2ImgPipeline

In [ ]:
dataset_dir = "/content/image-data-generation/dataset/source_road_scenes"
results_dir = "/content/image-data-generation/results/controlnet_vs_ipadapter"

# Uses a dedicated snow reference image for IP adapter.
reference_image_path = "/content/image-data-generation/dataset/weather_reference_conditions/snow_reference_1.jpg"

if not os.path.exists(reference_image_path):
    raise FileNotFoundError(f"Reference image not found: {reference_image_path}")

reference_image = Image.open(reference_image_path).convert("RGB").resize((512, 512))

print("Reference image loaded:", reference_image_path)

os.makedirs(results_dir, exist_ok=True)

image_paths = sorted([
    os.path.join(dataset_dir, f)
    for f in os.listdir(dataset_dir)
    if f.lower().endswith((".jpg", ".jpeg", ".png", ".jfif"))
])

if len(image_paths) == 0:
    raise FileNotFoundError("No road-scene images found in the dataset folder.")

print(f"Found {len(image_paths)} road-scene images.")

user_prompt = "Convert image to snowy winter with heavy snowfall"

prompt = (
    f"{user_prompt}. "
    "Preserve original road layout, vehicles, object positions, and perspective."
)

negative_prompt = (
    "blurry, distorted cars, distorted road, bad geometry, low quality, warped perspective"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("Device:", device)
print("Images:", image_paths)

In [ ]:
print("Loading MiDaS...")
midas_device = torch.device(device)

midas_model = torch.hub.load("intel-isl/MiDaS", "DPT_Hybrid")
midas_model.to(midas_device).eval()

midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.dpt_transform

print("Loading ControlNet...")
controlnet = ControlNetModel.from_pretrained(
    "lllyasviel/sd-controlnet-depth",
    torch_dtype=dtype
)

print("Loading Stable Diffusion + ControlNet...")
pipe = StableDiffusionControlNetImg2ImgPipeline.from_pretrained(
    "runwayml/stable-diffusion-v1-5",
    controlnet=controlnet,
    torch_dtype=dtype,
    safety_checker=None
).to(device)


In [ ]:
print("Loading IP-Adapter...")

pipe.load_ip_adapter(
    "h94/IP-Adapter",
    subfolder="models",
    weight_name="ip-adapter_sd15.bin"
)

pipe.set_ip_adapter_scale(0.3)

In [ ]:
for image_path in image_paths:
    image_id = os.path.splitext(os.path.basename(image_path))[0]
    output_dir = os.path.join(results_dir, image_id)
    os.makedirs(output_dir, exist_ok=True)

    print("\nProcessing:", image_id)

    image = Image.open(image_path).convert("RGB")
    init_image = image.resize((512, 512))
    init_image.save(os.path.join(output_dir, "original.png"))
    reference_image.save(os.path.join(output_dir, "reference_condition.png"))

    # MiDaS
    img_np = np.array(image)
    input_batch = transform(img_np).to(midas_device)

    with torch.no_grad():
        prediction = midas_model(input_batch)
        prediction = torch.nn.functional.interpolate(
            prediction.unsqueeze(1),
            size=img_np.shape[:2],
            mode="bicubic",
            align_corners=False
        ).squeeze(0).squeeze(0)

    depth = prediction.detach().cpu().numpy()
    depth_vis = cv2.normalize(depth, None, 0, 255, norm_type=cv2.NORM_MINMAX)
    depth_vis = depth_vis.astype(np.uint8)

    depth_gray = Image.fromarray(depth_vis).convert("RGB").resize((512, 512))
    depth_gray.save(os.path.join(output_dir, "depth_gray.png"))

    # ControlNet only, so reference image guidance is disabled through IP-Adapter
    pipe.set_ip_adapter_scale(0.0)

    generator = torch.Generator(device=device).manual_seed(42)

    start_time = time.time()
    controlnet_result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    controlnet_time = time.time() - start_time

    controlnet_result.save(os.path.join(output_dir, "controlnet_only.png"))

    # ControlNet + IP-Adapter
    pipe.set_ip_adapter_scale(0.3)

    generator = torch.Generator(device=device).manual_seed(42)

    start_time = time.time()
    ipadapter_result = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        image=init_image,
        control_image=depth_gray,
        ip_adapter_image=reference_image,
        strength=0.8,
        guidance_scale=7.5,
        controlnet_conditioning_scale=1.0,
        num_inference_steps=30,
        generator=generator
    ).images[0]
    ipadapter_time = time.time() - start_time

    ipadapter_result.save(os.path.join(output_dir, "controlnet_ipadapter.png"))

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))

    axes[0].imshow(init_image)
    axes[0].set_title("Original")
    axes[0].axis("off")

    axes[1].imshow(reference_image)
    axes[1].set_title("Weather Reference")
    axes[1].axis("off")

    axes[2].imshow(controlnet_result)
    axes[2].set_title("ControlNet Only")
    axes[2].axis("off")

    axes[3].imshow(ipadapter_result)
    axes[3].set_title("ControlNet + IP-Adapter")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(os.path.join(output_dir, "comparison.png"), dpi=300)
    plt.show()
    plt.close()

    # Save metadata to support traceability
    with open(os.path.join(output_dir, "metadata.txt"), "w") as f:
        f.write(f"Image ID: {image_id}\n")
        f.write(f"Input path: {image_path}\n")
        f.write(f"Reference image: {reference_image_path}\n")
        f.write(f"Prompt: {prompt}\n")
        f.write(f"Negative prompt: {negative_prompt}\n")
        f.write("Seed: 42\n")
        f.write("Image size: 512x512\n")
        f.write("Strength: 0.8\n")
        f.write("Guidance scale: 7.5\n")
        f.write("Inference steps: 30\n")
        f.write("ControlNet conditioning scale: 1.0\n")
        f.write("ControlNet-only IP-Adapter scale: 0.0\n")
        f.write("ControlNet + IP-Adapter scale: 0.3\n")
        f.write(f"ControlNet runtime: {controlnet_time:.2f} seconds\n")
        f.write(f"ControlNet + IP-Adapter runtime: {ipadapter_time:.2f} seconds\n")


    expected_files = [
        "original.png",
        "reference_condition.png",
        "depth_gray.png",
        "controlnet_only.png",
        "controlnet_ipadapter.png",
        "comparison.png",
        "metadata.txt"
    ]

    missing_files = [
        file for file in expected_files
        if not os.path.exists(os.path.join(output_dir, file))
    ]

    if missing_files:
        print("Missing files:", missing_files)
    else:
        print("All expected files saved for", image_id)

    gc.collect()
    torch.cuda.empty_cache()

print("All images processed.")

In [ ]:
del pipe
del controlnet
del midas_model
del midas_transforms
del transform

gc.collect()
torch.cuda.empty_cache()

print("GPU memory cleared.")
